In [1]:
import pandas as pd
import numpy as np
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')

master_df = pd.read_csv("../data/processed/master_features.csv")

print(f"Loaded {len(master_df)} companies")

Importing plotly failed. Interactive plots will not work.


Loaded 503 companies


In [2]:
ticker = "AAPL"

income_stmt = pd.read_csv(f"../data/raw/{ticker}/income_statement.csv", index_col=0)

revenue_row = income_stmt.loc['Total Revenue']
revenue_row = revenue_row.dropna()

prophet_df = pd.DataFrame({
    'ds': pd.to_datetime(revenue_row.index),
    'y': revenue_row.values.astype(float)
}).sort_values('ds').reset_index(drop=True)

print(prophet_df)

          ds             y
0 2022-09-30  3.943280e+11
1 2023-09-30  3.832850e+11
2 2024-09-30  3.910350e+11
3 2025-09-30  4.161610e+11


In [3]:
ticker_obj = yf.Ticker(ticker) if 'yf' in dir() else __import__('yfinance').Ticker(ticker)

import yfinance as yf
ticker_obj = yf.Ticker(ticker)

quarterly_income = ticker_obj.quarterly_financials

revenue_quarterly = quarterly_income.loc['Total Revenue'].dropna()

prophet_df_q = pd.DataFrame({
    'ds': pd.to_datetime(revenue_quarterly.index),
    'y': revenue_quarterly.values.astype(float)
}).sort_values('ds').reset_index(drop=True)

print(f"Quarterly data points: {len(prophet_df_q)}")
print(prophet_df_q)

Quarterly data points: 5
          ds             y
0 2025-03-31  9.535900e+10
1 2025-06-30  9.403600e+10
2 2025-09-30  1.024660e+11
3 2025-12-31  1.437560e+11
4 2026-03-31  1.111840e+11


In [5]:
def calculate_dcf(ticker, wacc_range=[0.08, 0.10, 0.12], terminal_growth_range=[0.02, 0.03, 0.04], forecast_years=5):
    try:
        income_stmt = pd.read_csv(f"../data/raw/{ticker}/income_statement.csv", index_col=0)
        cash_flow = pd.read_csv(f"../data/raw/{ticker}/cash_flow.csv", index_col=0)
        
        def get_val(df, possible_names):
            for name in possible_names:
                if name in df.index:
                    vals = df.loc[name].dropna()
                    if len(vals) > 0:
                        return vals.astype(float)
            return None

        revenue = get_val(income_stmt, ['Total Revenue'])
        fcf = get_val(cash_flow, ['Free Cash Flow'])
        
        if revenue is None or fcf is None or len(revenue) < 2:
            return None
        
        revenue = revenue.sort_index()
        fcf = fcf.sort_index()
        
        years = len(revenue)
        cagr = (revenue.iloc[-1] / revenue.iloc[0]) ** (1/(years-1)) - 1
        cagr = max(min(cagr, 0.30), -0.10)
        
        avg_fcf_margin = (fcf / revenue).mean()
        avg_fcf_margin = max(min(avg_fcf_margin, 0.40), 0.01)
        
        latest_revenue = revenue.iloc[-1]
        projected_revenues = [latest_revenue * (1 + cagr) ** i for i in range(1, forecast_years+1)]
        projected_fcf = [rev * avg_fcf_margin for rev in projected_revenues]
        
        results = {}
        for wacc in wacc_range:
            for tgr in terminal_growth_range:
                pv_fcfs = [fcf_val / (1 + wacc)**i for i, fcf_val in enumerate(projected_fcf, 1)]
                terminal_value = projected_fcf[-1] * (1 + tgr) / (wacc - tgr)
                pv_terminal = terminal_value / (1 + wacc)**forecast_years
                intrinsic_ev = sum(pv_fcfs) + pv_terminal
                results[(wacc, tgr)] = intrinsic_ev
        
        company_data = master_df[master_df['symbol'] == ticker].iloc[0]
        shares = company_data['shares_outstanding']
        net_debt = company_data['net_debt']
        
        intrinsic_values = {}
        for (wacc, tgr), ev in results.items():
            equity_value = ev - (net_debt if pd.notna(net_debt) else 0)
            intrinsic_values[(wacc, tgr)] = equity_value / shares if shares else None
        
        return {
            'ticker': ticker,
            'cagr': round(cagr * 100, 2),
            'avg_fcf_margin': round(avg_fcf_margin * 100, 2),
            'latest_revenue': latest_revenue,
            'projected_fcf': projected_fcf,
            'intrinsic_values': intrinsic_values,
            'base_case': intrinsic_values[(wacc_range[1], terminal_growth_range[1])]
        }
        
    except Exception as e:
        print(f"{ticker} DCF failed: {e}")
        return None

In [6]:
aapl_dcf = calculate_dcf("AAPL")

if aapl_dcf:
    print(f"Ticker: {aapl_dcf['ticker']}")
    print(f"Historical Revenue CAGR: {aapl_dcf['cagr']}%")
    print(f"Avg FCF Margin: {aapl_dcf['avg_fcf_margin']}%")
    print(f"Latest Revenue: ${aapl_dcf['latest_revenue']/1e9:,.1f}B")
    print(f"\nProjected FCF over 5 years:")
    for i, fcf in enumerate(aapl_dcf['projected_fcf'], 1):
        print(f"  Year {i}: ${fcf/1e9:,.1f}B")
    print(f"\nIntrinsic Value per Share — Sensitivity Table:")
    print(f"{'':>15}", end="")
    tgr_vals = sorted(set(k[1] for k in aapl_dcf['intrinsic_values'].keys()))
    for tgr in tgr_vals:
        print(f"  TGR={tgr*100:.0f}%", end="")
    print()
    wacc_vals = sorted(set(k[0] for k in aapl_dcf['intrinsic_values'].keys()))
    for wacc in wacc_vals:
        print(f"  WACC={wacc*100:.0f}%      ", end="")
        for tgr in tgr_vals:
            val = aapl_dcf['intrinsic_values'][(wacc, tgr)]
            print(f"  ${val:>8,.2f}  ", end="")
        print()
    print(f"\nBase Case (WACC=10%, TGR=3%): ${aapl_dcf['base_case']:,.2f}")

Ticker: AAPL
Historical Revenue CAGR: 1.81%
Avg FCF Margin: 26.45%
Latest Revenue: $416.2B

Projected FCF over 5 years:
  Year 1: $112.1B
  Year 2: $114.1B
  Year 3: $116.2B
  Year 4: $118.3B
  Year 5: $120.4B

Intrinsic Value per Share — Sensitivity Table:
                 TGR=2%  TGR=3%  TGR=4%
  WACC=8%        $  125.26    $  145.35    $  175.48  
  WACC=10%        $   93.70    $  103.70    $  117.03  
  WACC=12%        $   74.76    $   80.55    $   87.78  

Base Case (WACC=10%, TGR=3%): $103.70


In [7]:
import time

dcf_results = []
failed_dcf = []

for i, ticker in enumerate(master_df['symbol'].tolist()):
    result = calculate_dcf(ticker)
    if result:
        dcf_results.append({
            'symbol': result['ticker'],
            'cagr': result['cagr'],
            'avg_fcf_margin': result['avg_fcf_margin'],
            'dcf_base_case': result['base_case'],
            'dcf_bull': result['intrinsic_values'].get((0.08, 0.04)),
            'dcf_bear': result['intrinsic_values'].get((0.12, 0.02)),
        })
    else:
        failed_dcf.append(ticker)
    
    if (i+1) % 50 == 0:
        print(f"[{i+1}/503] processed...")

dcf_df = pd.DataFrame(dcf_results)
dcf_df.to_csv("../data/processed/dcf_valuations.csv", index=False)

print(f"\nDone. {len(dcf_results)} successful, {len(failed_dcf)} failed.")
print(f"Failed: {failed_dcf}")

[50/503] processed...
[100/503] processed...
[150/503] processed...
[200/503] processed...
[250/503] processed...
[300/503] processed...
[350/503] processed...
[400/503] processed...
[450/503] processed...
[500/503] processed...

Done. 501 successful, 2 failed.
Failed: ['BRK.B', 'BF.B']


In [8]:
dcf_df = pd.read_csv("../data/processed/dcf_valuations.csv")

current_prices = master_df[['symbol', 'current_price', 'name']].copy()
dcf_df = dcf_df.merge(current_prices, on='symbol', how='left')

dcf_df['dcf_upside'] = ((dcf_df['dcf_base_case'] - dcf_df['current_price']) / dcf_df['current_price'] * 100).round(1)

print(f"DCF results shape: {dcf_df.shape}")
print(f"\nSample — 10 most undervalued by DCF:")
under = dcf_df.nlargest(10, 'dcf_upside')[['symbol', 'name', 'current_price', 'dcf_base_case', 'dcf_upside']]
for _, row in under.iterrows():
    print(f"  {row['symbol']:<6} {str(row['name'])[:30]:<30} Current: ${row['current_price']:>8,.2f}  DCF: ${row['dcf_base_case']:>8,.2f}  Upside: {row['dcf_upside']:>+.1f}%")

print(f"\nSample — 10 most overvalued by DCF:")
over = dcf_df.nsmallest(10, 'dcf_upside')[['symbol', 'name', 'current_price', 'dcf_base_case', 'dcf_upside']]
for _, row in over.iterrows():
    print(f"  {row['symbol']:<6} {str(row['name'])[:30]:<30} Current: ${row['current_price']:>8,.2f}  DCF: ${row['dcf_base_case']:>8,.2f}  Upside: {row['dcf_upside']:>+.1f}%")

DCF results shape: (501, 9)

Sample — 10 most undervalued by DCF:
  ACGL   Arch Capital Group Ltd.        Current: $   91.22  DCF: $  780.58  Upside: +755.7%
  EG     Everest Group, Ltd.            Current: $  333.85  DCF: $2,626.74  Upside: +686.8%
  IBKR   Interactive Brokers Group, Inc Current: $   83.81  DCF: $  596.27  Upside: +611.5%
  COF    Capital One Financial Corporat Current: $  180.72  DCF: $  898.19  Upside: +397.0%
  SYF    Synchrony Financial            Current: $   70.87  DCF: $  344.83  Upside: +386.6%
  CINF   Cincinnati Financial Corporati Current: $  164.93  DCF: $  700.60  Upside: +324.8%
  APO    Apollo Global Management, Inc. Current: $  126.79  DCF: $  527.73  Upside: +316.2%
  MET    MetLife, Inc.                  Current: $   84.90  DCF: $  339.20  Upside: +299.5%
  PGR    The Progressive Corporation    Current: $  203.88  DCF: $  777.75  Upside: +281.5%
  CNC    Centene Corporation            Current: $   62.43  DCF: $  218.21  Upside: +249.5%

Sample — 10 m

In [9]:
ml_df = pd.read_csv("../data/processed/ml_valuations.csv")

final_df = master_df[['symbol', 'name', 'sector', 'current_price', 
                        'market_cap', 'pe_ratio', 'ev_ebitda', 
                        'altman_z_score', 'z_score_zone']].copy()

final_df = final_df.merge(
    ml_df[['symbol', 'fair_value_per_share', 'upside_pct']].rename(
        columns={'fair_value_per_share': 'ml_fair_value', 'upside_pct': 'ml_upside'}),
    on='symbol', how='left'
)

final_df = final_df.merge(
    dcf_df[['symbol', 'dcf_base_case', 'dcf_bull', 'dcf_bear', 'dcf_upside', 'cagr', 'avg_fcf_margin']],
    on='symbol', how='left'
)

final_df.to_csv("../data/processed/final_valuations.csv", index=False)

print(f"Final dataset shape: {final_df.shape}")
print(f"\nSample — Apple:")
aapl = final_df[final_df['symbol'] == 'AAPL'].iloc[0]
print(f"  Current Price:    ${aapl['current_price']:,.2f}")
print(f"  ML Fair Value:    ${aapl['ml_fair_value']:,.2f}  ({aapl['ml_upside']:+.1f}%)")
print(f"  DCF Base Case:    ${aapl['dcf_base_case']:,.2f}  ({aapl['dcf_upside']:+.1f}%)")
print(f"  Altman Z-Score:   {aapl['altman_z_score']:,.3f} ({aapl['z_score_zone']})")

Final dataset shape: (503, 17)

Sample — Apple:
  Current Price:    $312.40
  ML Fair Value:    $309.45  (-0.9%)
  DCF Base Case:    $103.70  (-66.8%)
  Altman Z-Score:   12.007 (safe)
